# Notebook 0 — Visite guidée du banc d'essai

## Le problème

Ce projet est un **banc d'essai chiffré** : chaque technique (chunking, hybrid search,
re-ranking, text-to-SQL, agents…) sera mesurée sur des jeux publics à vérité annotée,
et comparée aux scores publiés. Pourquoi cette contrainte ? Parce que sur un corpus
maison, un BM25 buggé et un BM25 correct produisent tous les deux « des résultats » —
et rien ne permet de les distinguer. Sur SciFact, un nDCG@10 très en dessous de la
littérature est un **bug détecté le jour même**.

Ce notebook fait deux choses :
1. montrer l'architecture cible et ce que chaque module fera ;
2. charger **réellement** chaque jeu de données du starter, regarder à quoi il
   ressemble, et comprendre ce qu'il mesure — et ce qu'il ne mesure pas.

**Contrat de tous les notebooks du projet** : rien n'est réimplémenté ici. Chaque
cellule importe le code de production de `src/llm_testbench/` et l'exécute. Le code
montré via `inspect.getsource()` est le code qui tourne.

## Expérimentations guidées

Modifie ces trois variables puis relance tout (`Run All`) :

- `BEIR_DATASET` : `"scifact"` ou `"nfcorpus"` — compare la taille des corpus et la
  forme des requêtes (revendications scientifiques vs questions médicales).
- `SHOW_N` : nombre d'exemples affichés par jeu.
- `INCLUDE_LARGE` : `True` pour télécharger aussi HotpotQA (1,3 Go) et
  BIRD Mini-Dev (~500 Mo, serveur à Pékin — patience).

In [1]:
BEIR_DATASET = "scifact"  # "scifact" | "nfcorpus"
SHOW_N = 2
INCLUDE_LARGE = False

## L'architecture cible

Un seul système, qui grossit à chaque phase :

| Module | Rôle | Arrive en |
|---|---|---|
| `llm_testbench.eval` | harness : loaders (✅ phase 0), métriques, juges, runners | phases 0–2 |
| `llm_testbench.llm` | abstraction provider, retries, cache, structured output | phase 1 |
| `llm_testbench.sources` | interface `Source` générique + connecteurs (docs, SQL, API, images) | phases 1 et 4 |
| `llm_testbench.ingest` | parsing, chunking, embeddings, indexation | phase 1 |
| `llm_testbench.app` | API FastAPI `/chat`, citations, refus explicite | phase 1 |
| `llm_testbench.retrieval` | hybrid search, re-ranking, transformation de requête | phase 3 |
| `llm_testbench.generation` | assemblage de contexte, chaînes | phases 1–3 |
| `llm_testbench.agents` | orchestration, outils, garde-fous, sécurité | phase 4 |
| `llm_testbench.obs` | instrumentation OTel/GenAI, coût, traces | phases 1 et 5 |

Ce qui existe réellement aujourd'hui :

In [2]:
from pathlib import Path

import llm_testbench

pkg_root = Path(llm_testbench.__file__).parent
for path in sorted(pkg_root.rglob("*.py")):
    rel = path.relative_to(pkg_root.parent)
    print(rel)

llm_testbench/__init__.py
llm_testbench/config.py
llm_testbench/eval/__init__.py
llm_testbench/eval/loaders/__init__.py
llm_testbench/eval/loaders/beir.py
llm_testbench/eval/loaders/bird_minidev.py
llm_testbench/eval/loaders/hotpotqa.py
llm_testbench/eval/loaders/qasper.py
llm_testbench/eval/loaders/spider.py
llm_testbench/eval/loaders/squad_v2.py
llm_testbench/eval/loaders/starter.py
llm_testbench/eval/loaders/tatqa.py
llm_testbench/eval/types.py


## La frontière du harness : les types partagés

Chaque jeu de données arrive dans un format brut différent (Parquet HF, JSON Aliyun,
JSON hub…). Les runners de la phase 2 ne verront **jamais** ces formats : ils
consommeront quatre familles de types Pydantic. C'est cette frontière qui permettra
d'ajouter un jeu sans toucher aux métriques, et inversement.

Les deux types les plus importants — lis les docstrings, elles portent les conventions :

In [3]:
import inspect

from llm_testbench.eval.types import QAExample, RetrievalDataset

print(inspect.getsource(RetrievalDataset))
print(inspect.getsource(QAExample))

class RetrievalDataset(BaseModel):
    """Corpus + requêtes + jugements de pertinence (format BEIR généralisé).

    ``qrels`` : query_id -> {doc_id: niveau de pertinence}. Un niveau > 0 signifie
    pertinent ; certains jeux ont des niveaux gradués (utile pour nDCG).
    """

    name: str
    documents: list[Document]
    queries: list[Query]
    qrels: dict[str, dict[str, int]]

class QAExample(BaseModel):
    """Question + contextes fournis par le jeu + réponses de référence.

    ``answers`` vide <=> ``is_answerable`` False : le comportement attendu du
    système est alors un refus explicite, et le harness mesurera le taux de
    refus correct / abusif.
    """

    example_id: str
    question: str
    contexts: list[Document] = Field(default_factory=list)
    answers: list[str] = Field(default_factory=list)
    is_answerable: bool
    # Ids des passages de preuve quand le jeu les annote (HotpotQA).
    supporting_doc_ids: list[str] = Field(default_factory=list)
    # Textes de 

## Le pattern de tous les loaders

**Les options.** (1) Un loader monolithique qui télécharge et parse d'un bloc —
simple, mais il faut le réseau pour tester le parsing, et le brief exige une suite de
tests hors ligne. (2) Séparer un **parsing pur** (des dicts bruts vers les types) d'un
**chargeur réseau mince** — le parsing se teste sur fixtures synthétiques, le
téléchargement reste trivial. On a pris (2), et c'est le pattern de tout le projet.

Exemple réel, l'assembleur BEIR — note le filtrage des requêtes non jugées :

In [4]:
from llm_testbench.eval.loaders.beir import build_retrieval_dataset

print(inspect.getsource(build_retrieval_dataset))

/Users/christophe/Sites/roadmap-ai-engineer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


def build_retrieval_dataset(
    name: str,
    corpus_rows: Iterable[Row],
    query_rows: Iterable[Row],
    qrels_rows: Iterable[Row],
) -> RetrievalDataset:
    """Assemblage pur (testable hors ligne) : filtre les requêtes non jugées."""
    qrels = parse_qrels(qrels_rows)
    queries = [q for q in parse_queries(query_rows) if q.query_id in qrels]
    return RetrievalDataset(
        name=f"beir/{name}",
        documents=parse_corpus(corpus_rows),
        queries=queries,
        qrels=qrels,
    )



---
## BEIR — le retrieval pur

**Ce que ça mesure** : la capacité à retrouver les bons documents (recall@k, nDCG@10 —
phase 2), avec des jugements de pertinence humains. **Ce que ça ne mesure pas** : la
qualité de la réponse générée — c'est exactement la séparation retrieval / génération
que le harness maintiendra partout.

Premier téléchargement : ~5 Mo pour SciFact, mis en cache dans `data/hf/`.

In [5]:
from llm_testbench.eval.loaders import load_beir

beir = load_beir(BEIR_DATASET)
print(f"{beir.name} : {len(beir.documents)} documents, {len(beir.queries)} requêtes jugées")

beir/scifact : 5183 documents, 300 requêtes jugées


Une requête, ses jugements, et le document jugé pertinent — la matière première de
toute métrique de retrieval :

In [6]:
query = beir.queries[0]
print("REQUÊTE :", query.text)
print("JUGEMENTS :", beir.qrels[query.query_id])

judged_doc_id = next(iter(beir.qrels[query.query_id]))
doc = next(d for d in beir.documents if d.doc_id == judged_doc_id)
print(f"\nDOCUMENT JUGÉ PERTINENT ({doc.doc_id}) — {doc.title}")
print(doc.text[:400], "…")

REQUÊTE : 0-dimensional biomaterials show inductive properties.
JUGEMENTS : {'31715818': 1}

DOCUMENT JUGÉ PERTINENT (31715818) — New opportunities: the use of nanotechnologies to manipulate and track stem cells.
Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the intracellular delivery of genes/oligonucleotides and protein/peptides; and engineered nanometer-scale scaffolds for stem ce …


---
## SQuAD 2.0 — l'abstention

**Ce que ça mesure** : le refus correct. ~la moitié des questions sont **non
répondables**, écrites pour ressembler à des répondables. Un système qui répond
toujours aura une excellente exactitude sur les répondables… et hallucinera sur le
reste. **Ce que ça ne mesure pas** : le retrieval (le contexte est fourni, un seul
paragraphe).

In [7]:
from llm_testbench.eval.loaders import load_squad_v2

squad = load_squad_v2()
unanswerable = [ex for ex in squad.examples if not ex.is_answerable]
answerable = [ex for ex in squad.examples if ex.is_answerable]
print(f"{len(squad.examples)} questions, dont {len(unanswerable)} non répondables "
      f"({100 * len(unanswerable) / len(squad.examples):.0f} %)")

11873 questions, dont 5945 non répondables (50 %)


In [8]:
for ex in [answerable[0], unanswerable[0]]:
    label = "RÉPONDABLE" if ex.is_answerable else "NON RÉPONDABLE (réponse attendue : refus)"
    print(f"[{label}]")
    print("  Contexte :", ex.contexts[0].text[:150], "…")
    print("  Question :", ex.question)
    print("  Réponses de référence :", ex.answers or "—")
    print()

[RÉPONDABLE]
  Contexte : The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a r …
  Question : In what country is Normandy located?
  Réponses de référence : ['France']

[NON RÉPONDABLE (réponse attendue : refus)]
  Contexte : The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a r …
  Question : Who gave their name to Normandy in the 1000's and 1100's
  Réponses de référence : —



---
## QASPER — les documents bruts entiers

**Ce que ça mesure** : c'est le seul jeu du starter qui fournit des **documents non
pré-découpés** (articles scientifiques complets, sectionnés) avec des questions
ancrées sur des passages de preuve. C'est le terrain du duel de stratégies de
chunking (phase 3) : si un chunking coupe la preuve en deux, ça se verra dans les
chiffres. **Ce que ça ne mesure pas** : la généralité (domaine NLP uniquement).

In [9]:
from llm_testbench.eval.loaders import load_qasper

papers = load_qasper()
n_questions = sum(len(p.questions) for p in papers)
n_unanswerable = sum(1 for p in papers for q in p.questions if not q.is_answerable)
print(f"{len(papers)} articles, {n_questions} questions, {n_unanswerable} non répondables")

281 articles, 1005 questions, 60 non répondables


In [10]:
paper = papers[0]
print("ARTICLE :", paper.title)
print("SECTIONS :", [s.section_name for s in paper.sections][:8], "…")
q = next(question for question in paper.questions if question.is_answerable)
print("\nQUESTION :", q.question)
print("RÉPONSES :", q.answers)
print("PREUVE ANNOTÉE :", q.evidence[0][:300] if q.evidence else "—")

ARTICLE : Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation
SECTIONS : ['Introduction', 'Related Work', 'Approach', 'Approach ::: Masked and Translation Language Model Pretraining', 'Approach ::: Bridge Language Model Pretraining', 'Approach ::: Transfer Protocol', 'Experiments ::: Setup', 'Experiments ::: Setup ::: Datasets.'] …

QUESTION : which multilingual approaches do they compare with?
RÉPONSES : ['BIBREF19', 'BIBREF20', 'multilingual NMT (MNMT) BIBREF19']
PREUVE ANNOTÉE : We compare our approaches with related approaches of pivoting, multilingual NMT (MNMT) BIBREF19, and cross-lingual transfer without pretraining BIBREF16. 


---
## Spider — le text-to-SQL propre

**Ce que ça mesure** : la génération SQL sur 20 bases de dev, avec la requête de
référence — et en phase 4, la vérité sera le **résultat d'exécution**, pas la
comparaison de chaînes. **Ce que ça ne mesure pas** : grand-chose de discriminant en
2026 — Spider est saturé (~86 % pour les modèles frontière) et gelé depuis 2024. Il
servira à valider la plomberie du harness ; le vrai signal viendra de BIRD.

In [11]:
from llm_testbench.eval.loaders import load_spider

spider = load_spider()
print(f"{len(spider.examples)} paires question→SQL sur le dev\n")
for ex in spider.examples[:SHOW_N]:
    print(f"[{ex.db_id}] {ex.question}")
    print(f"  → {ex.gold_sql}\n")

1034 paires question→SQL sur le dev

[concert_singer] How many singers do we have?
  → SELECT count(*) FROM singer

[concert_singer] What is the total number of singers?
  → SELECT count(*) FROM singer



---
## TAT-QA — tables financières + texte

**Ce que ça mesure** : le raisonnement numérique sur un croisement table + paragraphes.
Le piège central : la réponse vient avec une **échelle** (« thousand », « percent »…)
qui fait partie de la vérité. En phase 4, la règle absolue sera : l'arithmétique est
faite par du code, le modèle restitue — jamais l'inverse. **Ce que ça ne mesure pas** :
le retrieval (table et paragraphes sont fournis).

In [12]:
from llm_testbench.eval.loaders import load_tatqa

tatqa = load_tatqa()
arith = next(ex for ex in tatqa.examples if ex.answer_type == "arithmetic" and ex.scale)
print(f"{len(tatqa.examples)} questions (dev)\n")
print("TABLE (extrait) :")
for row in arith.table[:4]:
    print("  ", row)
print("\nQUESTION :", arith.question)
print(f"RÉPONSE : {arith.answers} — échelle : {arith.scale!r} — dérivation : {arith.derivation}")

1644 questions (dev)

TABLE (extrait) :
   ['', 'Years Ended December 31,', '', 'Change', '']
   ['', '2019', '2018', '$', '%']
   ['', '', '(dollars in thousands)', '', '']
   ['Impairment of goodwill', '$1,910', '$14,740', '$(12,830)', '(87%)']

QUESTION : What is the company's average goodwill impairment in 2018 and 2019?
RÉPONSE : ['8325'] — échelle : 'thousand' — dérivation : (14,740 + 1,910)/2 


Sans le champ `scale`, « 400 » serait compté juste alors que la vérité est
« 400 thousand » — ou l'inverse. Le scorer de la phase 4 devra normaliser les deux
côtés. C'est le genre de détail qui sépare un benchmark bien implémenté d'un
générateur de chiffres flatteurs.

---
## HotpotQA et BIRD Mini-Dev — les gros téléchargements

Passe `INCLUDE_LARGE = True` en tête de notebook pour les charger (1,3 Go + ~500 Mo).

- **HotpotQA (distractor)** : questions multi-hop, 10 paragraphes fournis dont 2 en
  or annotés — le retrieval est évaluable au niveau du paragraphe.
- **BIRD Mini-Dev** : le text-to-SQL réaliste et sale — 500 instances avec les bases
  SQLite, et un champ `evidence` (connaissance externe) qui fait partie du protocole.

In [13]:
if INCLUDE_LARGE:
    from llm_testbench.eval.loaders import load_bird_minidev, load_hotpotqa

    hotpot = load_hotpotqa()
    ex = hotpot.examples[0]
    print(f"hotpotqa : {len(hotpot.examples)} questions")
    print("  Question :", ex.question)
    print("  Contextes fournis :", len(ex.contexts), "— paragraphes en or :", ex.supporting_doc_ids)
    print("  Réponse :", ex.answers[0], "\n")

    bird = load_bird_minidev()
    bx = next(b for b in bird.examples if b.evidence)
    print(f"bird_minidev : {len(bird.examples)} instances — bases SQLite : {bird.databases_dir}")
    print("  Question :", bx.question)
    print("  Evidence :", bx.evidence)
    print("  SQL or   :", bx.gold_sql)
else:
    print("INCLUDE_LARGE = False — HotpotQA et BIRD Mini-Dev non téléchargés ici.")

INCLUDE_LARGE = False — HotpotQA et BIRD Mini-Dev non téléchargés ici.


---
## Récapitulatif : qui mesure quoi

| Jeu | Mesure | Ne mesure pas | Phase |
|---|---|---|---|
| SciFact / NFCorpus | retrieval (nDCG, recall@k) | la génération | 2–3 |
| SQuAD 2.0 | refus correct / refus abusif | le retrieval | 2 |
| QASPER | chunking sur documents bruts, preuves | la généralité hors NLP | 2–3 |
| HotpotQA | multi-hop, retrieval de paragraphes | le multi-hop « dur » (→ MuSiQue) | 2–3 |
| Spider | plomberie text-to-SQL | la difficulté réelle (saturé) | 4 |
| BIRD Mini-Dev | text-to-SQL réaliste, exécution | — | 4 |
| TAT-QA | raisonnement numérique table+texte | le retrieval | 4 |

Arriveront avec leurs phases : MTEB (embeddings, phase 1), MuSiQue (phase 2–3),
tau2-bench, BFCL, AgentDojo, ChartQA, CORD, DocVQA (phase 4).

## Les pièges (ce que les tutoriels ne disent pas)

1. **Les qrels BEIR vivent dans des repos séparés** (`BeIR/scifact-qrels`) et la
   majorité des requêtes du fichier `queries` ne sont pas jugées : sans filtrage, les
   métriques de la phase 2 seraient diluées par des requêtes inévaluables.
2. **Le marqueur « non répondable » de SQuAD 2.0 est implicite** (liste de réponses
   vide) — facile à confondre avec une donnée manquante.
3. **BIRD sans le champ `evidence` s'effondre** : ce n'est pas de la triche, c'est le
   protocole officiel ; l'oublier rend les scores incomparables à la littérature.
4. **QASPER a plusieurs annotations par question, parfois contradictoires** — notre
   convention (non répondable = unanimité des annotateurs) est documentée dans le
   code, parce qu'elle change les chiffres.
5. **L'échelle TAT-QA fait partie de la réponse** — un scorer naïf compte faux des
   résultats justes.
6. **Contamination** : ces jeux sont probablement dans les données d'entraînement des
   modèles. Les scores seront flattés — mesuré et documenté en phase 2.5.

## Les chiffres

Pas encore de chiffres de qualité — c'est le rôle de la baseline de la phase 2. Les
volumétries ci-dessus (documents, requêtes jugées, taux de non-répondables) sont les
seuls chiffres honnêtes disponibles à ce stade.

## Questions d'entretien

<details><summary><b>1. Pourquoi évaluer le retrieval et la génération séparément ?</b></summary>

Parce qu'une métrique bout-en-bout ne dit pas **où** est le problème : une mauvaise
réponse peut venir d'un contexte raté (retrieval) ou d'un contexte mal utilisé
(génération). On mesure le retrieval avec des jugements de pertinence (recall@k,
nDCG), puis la génération **à contexte supposé parfait** (groundedness, citations,
refus). Sinon on optimise à l'aveugle le mauvais étage.
</details>

<details><summary><b>2. Votre benchmark donne 15 points de moins que la littérature. Réaction ?</b></summary>

C'est un bug jusqu'à preuve du contraire, pas un résultat. Vérifier dans l'ordre : le
protocole (mêmes splits ? mêmes qrels ? champ `evidence` fourni pour BIRD ?), le
parsing, la métrique (nDCG@10 vs @100), puis seulement l'implémentation. Un écart
important non expliqué invalide toute comparaison ultérieure.
</details>

<details><summary><b>3. Pourquoi un jeu avec questions non répondables est-il indispensable ?</b></summary>

Sans lui, le taux d'hallucination est invisible : un système qui répond toujours
maximise l'exactitude sur les répondables. SQuAD 2.0 permet de mesurer deux erreurs
distinctes — répondre quand il fallait refuser, refuser quand il fallait répondre —
qui se règlent par des leviers différents.
</details>

<details><summary><b>4. Pourquoi ne pas évaluer sur vos propres documents métier ?</b></summary>

Aucun repère : sans score publié, impossible de distinguer « corpus difficile » d'
« implémentation fausse ». Les jeux publics donnent un étalon ; le golden set maison
(50–80 questions, phase 2) démontre la **méthode** de construction d'une éval, pas le
volume.
</details>

<details><summary><b>5. Les scores publics sont-ils fiables vu la contamination des données d'entraînement ?</b></summary>

Non, pas aveuglément : les jeux anciens sont probablement vus à l'entraînement, donc
les scores absolus sont flattés. Ils restent utiles en **relatif** (comparer nos
techniques entre elles) et comme détecteur de bug. Mesure prévue : signaux de
contamination + petit jeu « frais » postérieur aux cutoffs, écart documenté.
</details>

## Ce qu'on n'a pas fait, et pourquoi

- **Pas de métriques** : elles arrivent avec le harness (phase 2) — les brancher
  maintenant, sans système à mesurer, produirait des chiffres sans objet.
- **Pas de MuSiQue / 2Wiki / jeux agents / jeux images** : chaque loader arrive avec
  la phase qui le consomme, pour éviter le code mort non testé en conditions réelles.
- **Pas de conversion Spider/BIRD vers Postgres** : c'est un vrai chantier (sqlglot),
  planifié en phase 4 avec son notebook dédié.
- **Pas d'appel LLM** : ce notebook tourne dans la CI à coût zéro et sans secrets.
  Le premier appel LLM (phase 1) introduira le mode replay pour que ça reste vrai.

**Prochaine étape (phase 1)** : la couche LLM robuste et l'interface `Source` — et le
premier notebook avec de vraies sorties intermédiaires d'appels LLM.